<p> <center> <a href="../start_here.ipynb">Home Page</a> </center> </p>

<div>
    <span style="float: left; width: 33%; text-align: left;"><a href="04_langraph_agent.ipynb">Previous Notebook</a></span>
    <span style="float: left; width: 34%; text-align: center;">
        <a href="01_inference_endpoint.ipynb">1</a>
        <a href="02_introduction_mcp.ipynb">2</a>
        <a href="03_low_level_mcp.ipynb">3</a>
        <a href="04_langraph_agent.ipynb">4</a>
        <a >5</a>
        <a href="06_challenge.ipynb">6</a>
        <a href="bonus_challenge/07_bonus_challenge.ipynb">7</a>
    </span>
    <span style="float: left; width: 33%; text-align: right;"><a href="06_challenge.ipynb">Next Notebook</a></span>
</div>

## Learning Objectives

By the end of this notebook, you will be able to:
- Build a Movie Database MCP Server using the low-level SDK with SQLite
- Use the `nat mcp client` CLI to discover and test MCP tools without writing client code
- Define NAT workflow configs in YAML to connect MCP tools with NVIDIA NIM endpoints
- Wrap a LangGraph intent classifier inside a NAT workflow to route queries before they reach the agent
- Configure NAT telemetry with Phoenix for end-to-end tracing

## Introduction

In the last notebook, we built a LangGraph agent by hand — wiring the LLM, tools, and control flow in Python. That works, but it doesn't scale: every new agent means more glue code, and there's no standard way to share tools across agents or swap LLM backends.

The **NVIDIA NeMo Agent Toolkit (NAT)** is designed to solve exactly that. It's a framework-agnostic toolkit where agents, LLMs, and tools are declared in YAML, connected through the Model Context Protocol (MCP), and run through a single CLI (`nat run`, `nat serve`, `nat eval`). The same YAML config can run a ReAct agent, a LangGraph workflow, or a multi-agent system — and swap Nemotron for GPT or Llama without touching the agent code.

In this notebook, you'll build a small but complete NAT-powered system end to end:

1. A **SQLite-backed Movie Database** (117 movies with IMDB ratings, box office earnings, and genres across 3 tables)
2. Exposed as a **low-level MCP Server** with a `search_movies` tool
3. Consumed by a **NAT ReAct agent** configured entirely in YAML
4. Gated by a **LangGraph intent classifier** that routes in-domain queries to the agent and rejects everything else
5. Traced end-to-end with **Phoenix telemetry**

By the end, you'll have the same architectural pattern that powers NVIDIA's production agentic systems — just at notebook scale.

## Setup Environment

Before we build anything, we need an LLM backend. NAT can talk to any NVIDIA NIM endpoint — hosted on `build.nvidia.com` or running locally on your own GPU. We'll default to the cloud endpoint for speed, but the local path is documented below if you want to run fully offline.

### Cloud Endpoint

The cell below will prompt for your key if `NVIDIA_API_KEY` isn't already set. See [Notebook 1](01_inference_endpoint.ipynb) if you haven't generated one yet.

In [ ]:
import os
import getpass
import warnings
warnings.filterwarnings("ignore")
if not os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"):
    nvapi_key = getpass.getpass("Enter your NVIDIA API key: ")
    assert nvapi_key.startswith("nvapi-"), f"{nvapi_key[:5]}... is not a valid key"
    os.environ["NVIDIA_API_KEY"] = nvapi_key
    os.environ["NGC_API_KEY"] = nvapi_key

### Local Endpoint

If you'd rather run the LLM locally instead of hitting the cloud endpoint, you can point NAT at any NIM container running on your machine. See [Notebook 1](01_inference_endpoint.ipynb) for how to launch one — the only change required later is swapping the `base_url` in the workflow YAML.

## Explore the Database

Before we expose the data as MCP tools, let's get familiar with what's actually in `movie.sqlite`. The database has three tables:

## Preview Some Rows

Let's preview some rows from the IMDB table to understand the data structure — each movie has an ID, title, rating, vote count, budget, and runtime.

In [ ]:
import sqlite3
import json

DB_PATH = "data/movie.sqlite"

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cursor = conn.cursor()
cursor.execute("SELECT Movie_id, Title, Rating, TotalVotes, Budget, Runtime FROM IMDB LIMIT 5")
rows = [dict(row) for row in cursor.fetchall()]
conn.close()

print(json.dumps(rows, indent=2))

### Wrap the Database in a Helper Class

We create a `MovieDB` helper class that wraps SQLite queries and returns results as MCP-typed `TextContent` objects. This pattern keeps the database logic separate from the MCP server, making it easy to test independently.

The `%%writefile` magic writes the cell contents to a Python file so it can be imported by both the notebook and the MCP server.

In [ ]:
%%writefile movie_db.py
import sqlite3
import json                                                                                                                                                                             
from typing import List
from pathlib import Path                                                                                                                                                                
import mcp.types as types

DB_PATH = "data/movie.sqlite"

class MovieDB:
  def __init__(self, db_path):
      self.db_path = str(Path().resolve().joinpath(db_path))

  def _search_movies(
      self,
      title: str | None,
      min_rating: float | None,
      max_rating: float | None,
      limit: int = 20,
  ) -> List[types.TextContent]:
      """Search movies in the IMDB table by title and/or rating range.

      Args:
          title: Partial or full movie title to search for.
          min_rating: Minimum IMDB rating.
          max_rating: Maximum IMDB rating.
          limit: Max number of results to return.
      """

      conn = sqlite3.connect(self.db_path)
      cursor = conn.cursor()

      query = """
          SELECT Movie_id, Title, Rating, TotalVotes, Budget, Runtime
          FROM IMDB
      """
      params = []
      conditions = []

      if title is not None:
          conditions.append("Title LIKE ?")
          params.append(f"%{title}%")

      if min_rating is not None:
          conditions.append("Rating >= ?")
          params.append(min_rating)

      if max_rating is not None:
          conditions.append("Rating <= ?")
          params.append(max_rating)

      if conditions:
          query += " WHERE " + " AND ".join(conditions)

      query += " ORDER BY Rating DESC LIMIT ?"
      params.append(limit)

      try:
          cursor.execute(query, params)
          results = cursor.fetchall()

          output = []
          for row in results:
              output.append(
                  {
                      "movie_id": row[0],
                      "title": row[1],
                      "rating": row[2],
                      "total_votes": row[3],
                      "budget": row[4],
                      "runtime": row[5],
                  }
              )

      except sqlite3.Error as e:
          conn.close()
          raise e

      finally:
          conn.close()

      return [types.TextContent(
          type="text",
          text=json.dumps(output)
      )]

In [ ]:
from movie_db import MovieDB

In [ ]:
movie_db = MovieDB(DB_PATH)

### Test Helper Class

Before building the MCP server, let's verify the `MovieDB` class works correctly with a few different query patterns. Each call returns a one-element list containing an MCP `TextContent(type="text", text=…)`, where `text` is the JSON-encoded result list — that's the shape the MCP client on the other end expects.

**Search by title** — find movies with "Dark" in the title:

In [ ]:
movie_db._search_movies(title="Dark", min_rating=None, max_rating=None)

**Search by minimum rating** — movies rated 8.5 or higher:

In [ ]:
movie_db._search_movies(title=None, min_rating=8.5, max_rating=None, limit=5)

**Combined filters** — movies with "The" in the title and rating above 8.0:

In [ ]:
movie_db._search_movies(title="The", min_rating=8.0, max_rating=None, limit=3)

**No filters** — return the top 5 movies by rating:

In [ ]:
movie_db._search_movies(title=None, min_rating=None, max_rating=None, limit=5)

## Build the MCP Server

Now we wrap our `MovieDB` class in a low-level MCP server with HTTP transport (similar to what we built in [Notebook 3](03_low_level_mcp.ipynb)). The server exposes a single `search_movies` tool that clients can discover and call. We use Starlette + Uvicorn to serve the MCP endpoint at `http://127.0.0.1:8000/mcp`.

In [ ]:
%%writefile movie_server.py                                                                                                                                                             
import sys
import os
from typing import Any                                                                                                                                                                  
from mcp.server.lowlevel import Server                                                                                                                                                  
from mcp.server.streamable_http_manager import StreamableHTTPSessionManager
from starlette.applications import Starlette                                                                                                                                            
from starlette.routing import Mount
from starlette.types import Receive, Scope, Send
import mcp.types as types
import json
import contextlib
import logging
from collections.abc import AsyncIterator
import uvicorn
from movie_db import MovieDB

logger = logging.getLogger(__name__)

DB_PATH = "data/movie.sqlite"


def main(db_path: str = DB_PATH):
  movie_db = MovieDB(db_path)
  mcp = Server("movie-db")

  @mcp.list_tools()
  async def handle_list_tools() -> list[types.Tool]:
      return [
          types.Tool(
      name="search_movies",                                                                                                                                                               
      description="Search movies by title and/or rating range. Results are sorted by rating descending. To get the top N highest rated movies, just set limit=N without any rating filters. Rating scale is 1.0 to 10.0. In this dataset, ratings range from 7.5 to 8.8.",
      inputSchema={
          "type": "object",
          "properties": {
              "title": {"type": "string", "description": "Partial or full movie title to search for"},
              "min_rating": {"type": "number", "description": "Minimum IMDB rating (1.0 to 10.0)"},
              "max_rating": {"type": "number", "description": "Maximum IMDB rating (1.0 to 10.0)"},
              "limit": {"type": "integer", "description": "Max results to return (default 20)"},
          },
      },
      ),
      ]

  @mcp.call_tool()
  async def handle_call_tool(name: str, args: dict[str, Any] | None):
      args = args or {}

      if name == "search_movies":
          return movie_db._search_movies(
              title=args.get("title"),
              min_rating=args.get("min_rating"),
              max_rating=args.get("max_rating"),
              limit=args.get("limit", 20),
          )
      else:
          return [types.TextContent(
              type="text",
              text=json.dumps({"error": f"Unknown tool: {name}"})
          )]

  session_manager = StreamableHTTPSessionManager(
      app=mcp,
      event_store=None,
      json_response=True,
      stateless=True,
  )

  async def handle_streamable_http(
      scope: Scope, receive: Receive, send: Send
  ) -> None:
      await session_manager.handle_request(scope, receive, send)

  @contextlib.asynccontextmanager
  async def lifespan(app: Starlette) -> AsyncIterator[None]:
      async with session_manager.run():
          logger.info("Movie DB MCP server started!")
          try:
              yield
          finally:
              logger.info("Movie DB MCP server shutting down...")

  starlette_app = Starlette(
      debug=True,
      routes=[
          Mount("/mcp", app=handle_streamable_http),
      ],
      lifespan=lifespan,
  )
  port = int(os.environ.get("MCP_PORT", "8000"))
  uvicorn.run(starlette_app, host="127.0.0.1", port=port)


if __name__ == "__main__":
  db_path = sys.argv[1] if len(sys.argv) > 1 else DB_PATH
  main(db_path)

Start the MCP server in the background so we can test it from subsequent cells.

In [ ]:
import os, subprocess, time

MCP_PORT = int(os.environ["MCP_PORT"])

log = open("mcp_server.log", "w")
mcp_server_process = subprocess.Popen(
    ["python", "movie_server.py", "data/movie.sqlite"],
    stdout=log,
    stderr=subprocess.STDOUT,
)

time.sleep(2)  # give the server a moment to bind the port
print(f"Server started on http://127.0.0.1:{MCP_PORT}/mcp (PID: {mcp_server_process.pid})")
print("Logs: mcp_server.log")

> **Tip:** If the server fails to start, check `mcp_server.log` in the working directory for errors. The most common cause is the port already being in use from a previous session.

## Test with NAT CLI

`nat` is the single entry point that ships with the NeMo Agent Toolkit. Instead of importing Python modules and wiring runtimes by hand, you point `nat` at a YAML config and it loads the LLMs, tool servers, agents, and telemetry declared in that file. The same binary covers the full lifecycle of an agentic system:

| Command | What it does |
|---------|-------------|
| `nat run --config_file workflow.yml --input "..."` | Execute a workflow once against a single input (we'll use this below). |
| `nat serve --config_file workflow.yml` | Expose the workflow as an HTTP/SSE API so other services can call it. |
| `nat eval --config_file workflow.yml` | Run the workflow against a dataset with the built-in evaluation harness. |
| `nat mcp client tool list --url ...` | Connect to any MCP server, run the `tools/list` RPC, and print every tool with its JSON schema. |
| `nat mcp client tool call <name> --url ... --json-args '{...}'` | Invoke a named MCP tool and print the raw response. |
| `nat info components` | List every plugin (`_type`) available for use in YAML configs — agents, LLMs, retrievers, exporters, etc. |

The `nat mcp client` subcommand is what we need right now: it replaces the hand-written MCP client from [Notebook 3](03_low_level_mcp.ipynb) with two one-liners. We'll use them to confirm (1) the server advertises `search_movies` with the schema we defined, and (2) it returns the same rows as the direct `movie_db._search_movies(...)` calls above. If either step fails, the agent in the next section would fail for the same reason — better to catch it here.

In [ ]:
!nat mcp client tool list --url http://127.0.0.1:{MCP_PORT}/mcp/

Now call the `search_movies` tool directly via the CLI to verify it returns the expected results.

In [ ]:
!nat mcp client tool call search_movies \
      --url http://127.0.0.1:${MCP_PORT}/mcp/ \
      --json-args '{"min_rating": 8.5, "limit": 5}'

## Define a NAT Workflow

NAT workflows are defined in YAML configuration files. A workflow config specifies three main sections:
- **`function_groups`**: External tool connections (here, our MCP server)
- **`llms`**: LLM backends to use (here, NVIDIA NIM with Nemotron)
- **`workflow`**: The agent type and how tools connect to the LLM

We'll start with a simple ReAct agent that can call our movie search tool. The `parse_agent_response_max_retries: 3` knob below tells the agent to retry up to three times if its own output can't be parsed as a valid ReAct `Action:` / `Thought:` block — a common failure mode when the LLM wanders off-format.

In [ ]:
%%writefile movie_workflow.yml                                                                                                                                                          
function_groups:                                                                                                                                                                        
    mcp_movies:                                                                                                                                                                           
      _type: mcp_client                                                                                                                                                                   
      server:                                                                                                                                                                             
        transport: streamable-http
        url: "http://127.0.0.1:${MCP_PORT}/mcp/"

llms:
    nim_llm:
      _type: nim
      model_name: nvidia/nemotron-3-nano-30b-a3b
      base_url: https://integrate.api.nvidia.com/v1
      api_key: ${NVIDIA_API_KEY}
      temperature: 0.0
      max_tokens: 1024

workflow:
    _type: react_agent
    tool_names:
      - mcp_movies
    llm_name: nim_llm
    verbose: true
    parse_agent_response_max_retries: 3

Run the workflow with a simple movie query.

In [ ]:
!nat run --config_file movie_workflow.yml --input "movies rated over 8.5"

### Probing the Agent's Boundaries

The in-domain query worked. But this bare ReAct agent has **no gate** on what counts as "in-domain" — nothing stops it from reaching for `search_movies` on questions that have nothing to do with the movie database. Let's probe its behavior on two clearly out-of-domain prompts and an in-domain factual query.

In [ ]:
!nat run --config_file movie_workflow.yml --input "How do I cook pasta?"

In [ ]:
!nat run --config_file movie_workflow.yml --input "What is the capital of France?"

In [ ]:
!nat run --config_file movie_workflow.yml --input "what is the rating of the movie The Dark Knight Rises?"

Depending on the run, you'll see a mix of behaviors across the three inputs:

- **Refusals** — the agent correctly declines (good, but not guaranteed)
- **Spurious tool calls** — `search_movies` invoked with `title="pasta"` or `title="France"`, burning tokens and tool latency for nothing
- **Hallucinations** — an answer invented without any tool call at all
- **Retry churn** — the ReAct loop exhausting its `parse_agent_response_max_retries` budget before giving up

None of this is a bug in NAT or in the agent — it's just what happens when you let arbitrary user input flow straight into a tool-calling loop. The in-domain query at the end (*Dark Knight Rises*) probably worked, but by the time we've run all three we have no principled way to tell *why* each one succeeded or failed without re-reading the verbose logs.

For a production system we want two things:

1. A **deterministic gate** — every query is classified first, and only in-domain ones reach the agent.
2. A way to **see what happened** — structured traces we can filter and diff, not a wall of text.

The next two sections add both: a LangGraph intent classifier (the gate) and Phoenix telemetry (the view into what the gate and agent actually did).

## Build an Intent Classifier with LangGraph

The behavior you just observed is common: a tool-using agent will try to use its tools, even on queries that clearly don't belong in its domain. The fix is to put a **cheap, deterministic classifier** in front of the agent so that only relevant questions pay for a full ReAct loop.

LangGraph fits naturally here because the routing is a small state machine:

Two things make this more than prompt engineering:

1. **Separation of concerns** — the classifier LLM sees only the question, never the tool descriptions, so it's much harder to confuse into thinking every input needs a tool.
2. **Cost control** — rejected queries never reach the agent loop and never touch MCP; `reject` is a hard-coded string with zero LLM cost.

### What's in the File

The next cell writes ~50 lines of Python. Here's what each piece does so the code reads cleanly:

1. **State type — `ConversationState`**
    ```python
    class ConversationState(TypedDict):
      messages: Annotated[list, add_messages]
    ```
    A typed dict with one field. The `Annotated[list, add_messages]` is a **LangGraph reducer** — it tells LangGraph that when multiple nodes return `{"messages": [...]}`, the new messages get *appended* to the running list instead of replacing it. That's why every node below returns a dict with a `messages` list.

2. **Two handles from the NAT runtime**
    ```python
    chat_llm = SyncBuilder.current().get_llm("nim_llm", wrapper_type=LLMFrameworkEnum.LANGCHAIN)
    movie_agent = SyncBuilder.current().get_function("movie_agent")
    ```
    These run at module load. `SyncBuilder.current()` reaches into the active NAT process and pulls the NIM LLM and the `movie_agent` ReAct function by the names registered in the YAML. Zero manual instantiation, zero API key handling in code — NAT does it once and any node that needs them just asks.

3. **`classify_node`** — one prompt, one LLM call. Notice the return shape: the classifier's verdict (`"in_domain"` or `"out_of_domain"`) is written back into the conversation as an *assistant message*. This is a tidy convention — the routing decision lives in the shared `messages` state, so downstream nodes (and Phoenix traces) can see it without adding a parallel "verdict" field.

4. **`route_intent`** — a conditional edge. LangGraph's `add_conditional_edges` expects a function that returns a node name; this one reads the last message (the classifier's verdict) and returns `"movie_agent"` or `"reject"`. Pure string logic, so it's **sync**.

5. **`movie_agent_node`** is `async def`, `reject_node` is plain `def`. The ReAct agent exposes `ainvoke` (NAT `Function` objects are awaitable), so any node that calls it must be async. `reject_node` just returns a hardcoded string, so sync is fine. LangGraph runs the mix without complaint.

In [ ]:
%%writefile movie_intent_classifier.py
from typing import Annotated
from typing_extensions import TypedDict                                                                                                                                                 
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

from nat.builder.framework_enum import LLMFrameworkEnum
from nat.builder.sync_builder import SyncBuilder

chat_llm = SyncBuilder.current().get_llm("nim_llm", wrapper_type=LLMFrameworkEnum.LANGCHAIN)
movie_agent = SyncBuilder.current().get_function("movie_agent")


class ConversationState(TypedDict):
  messages: Annotated[list, add_messages]


def classify_node(state: ConversationState):
  response = chat_llm.invoke([
      {"role": "system", "content": (
          "You are an intent classifier for a movie database assistant. "
          "The database contains IMDB movie data: titles, ratings, votes, budgets, and runtimes. "
          "Classify the user's question as either 'in_domain' or 'out_of_domain'. "
          "Reply with ONLY one word.\n\n"
          "in_domain: questions about movies, ratings, top movies, searching movies by title, "
          "comparing movie ratings, budgets, runtimes.\n\n"
          "out_of_domain: anything unrelated to movies."
      )},
      *state["messages"]
  ])
  raw = response.content.strip().lower()
  return {"messages": [{"role": "assistant", "content": raw}]}


def route_intent(state: ConversationState):
  last = state["messages"][-1].content.strip().lower()
  if last == "in_domain":
      return "movie_agent"
  return "reject"


async def movie_agent_node(state: ConversationState):
  user_query = state["messages"][0].content
  result = await movie_agent.ainvoke(user_query)
  return {"messages": [{"role": "assistant", "content": str(result)}]}


def reject_node(state: ConversationState):
  return {"messages": [{"role": "assistant", "content": "Sorry, I can only help with movie-related questions."}]}


graph = StateGraph(ConversationState)
graph.add_node("classify", classify_node)
graph.add_node("movie_agent", movie_agent_node)
graph.add_node("reject", reject_node)

graph.add_edge(START, "classify")
graph.add_conditional_edges("classify", route_intent, {"movie_agent": "movie_agent", "reject": "reject"})
graph.add_edge("movie_agent", END)
graph.add_edge("reject", END)

intent_graph = graph.compile()

The classifier also gives us a clean place to hang telemetry later: each node (`classify`, `movie_agent`, `reject`) emits its own span, so in the next section we'll be able to *see* which path every query took instead of guessing.

In [ ]:
%%writefile movie_intent_workflow.yml                                                                                                                                                   
function_groups:                                                                                                                                                                        
    mcp_movies:                                                                                                                                                                           
      _type: mcp_client
      server:                                                                                                                                                                             
        transport: streamable-http
        url: "http://127.0.0.1:${MCP_PORT}/mcp/"

llms:
    nim_llm:
      _type: nim
      model_name: nvidia/nemotron-3-nano-30b-a3b
      base_url: https://integrate.api.nvidia.com/v1
      api_key: ${NVIDIA_API_KEY}
      temperature: 0.0
      max_tokens: 1024

functions:
    movie_agent:
      _type: react_agent
      tool_names:
        - mcp_movies
      llm_name: nim_llm
      verbose: true
      parse_agent_response_max_retries: 3

workflow:
    _type: langgraph_wrapper
    graph: movie_intent_classifier.py:intent_graph

In [ ]:
!nat run --config_file movie_intent_workflow.yml --input "How do I cook pasta?"

In [ ]:
!nat run --config_file movie_intent_workflow.yml --input "What is the capital of France?"

In [ ]:
!nat run --config_file movie_intent_workflow.yml --input "what is the rating of the movie The Dark Knight Rises?"

## Telemetry with Phoenix

Agents are harder to debug than traditional software. A single user query can fan out into multiple LLM calls, tool invocations, and reasoning steps — each with its own latency, cost, and failure mode. When the agent returns a wrong answer, `print()` tells you *what* came out, but not *why*: which tool returned garbage? Which LLM step hallucinated? Did the ReAct loop exhaust its retries before landing on a coherent plan?

This is what **observability** solves. Instead of scattered `print` statements, we emit structured **spans** that capture every step of execution — inputs, outputs, timing, token counts, errors — and visualize them as a trace tree. That trace becomes your debugger, profiler, and evaluation dataset in one.

### Why Phoenix?

[Arize Phoenix](https://phoenix.arize.com/) is an open-source LLM observability platform built on two open standards:

- **OpenTelemetry (OTel)** — the industry-standard wire protocol for distributed tracing, already widely deployed across production microservice infrastructure, so the data model is well understood by engineering teams.
- **OpenInference** — a semantic convention layered on top of OTel that standardizes *how* LLM/agent concepts (chat messages, tool calls, retrievals, embeddings, reranker scores) are represented as span attributes.

Because NAT and Phoenix both speak OpenInference, telemetry is portable: the same trace viewer renders a NAT ReAct agent, a LangGraph workflow, and a CrewAI crew without custom adapters. And because the wire format is OTel, the same spans can be routed to any compatible backend by swapping the exporter config — no agent code changes.

### What NAT Instruments Automatically

Once Phoenix is enabled in the workflow YAML, every layer of the stack emits spans — no decorators, no SDK calls, no `start_span` / `end_span` bookkeeping:

| Layer | Captured as span attributes |
|-------|-----------------------------|
| **Workflow entry** | User input, final output, wall-clock latency |
| **LangGraph nodes** | Node name, state in/out, routing decision |
| **ReAct iterations** | Thought, action chosen, observation, iteration index |
| **LLM calls** | Model name, full message history, completion, prompt/completion tokens, finish reason |
| **Tool calls (incl. MCP)** | Tool name, JSON arguments, return value, duration, exceptions |

The result is a nested waterfall: the outermost span is the user query, its children are graph nodes, their children are LLM and tool calls, all the way down to the raw MCP request on the wire.

### Starting the Phoenix Observability Server

Before we build and run our agent, we'll launch [Arize Phoenix](https://docs.arize.com/phoenix) — an open-source observability platform for LLM applications. Phoenix will capture **traces** from our NeMo Agent Toolkit workflow, letting us inspect every LLM call, tool invocation, and agent decision step in a visual UI.

In [ ]:
import os, subprocess

PHOENIX_PORT = int(os.environ["PHOENIX_PORT"])

log = open("phoenix.log", "w")
phoenix_process = subprocess.Popen(
    ["phoenix", "serve"],
    stdout=log,
    stderr=subprocess.STDOUT,
)

print(f"Server started on http://127.0.0.1:{PHOENIX_PORT} (PID: {phoenix_process.pid})")
print(f"Access via port-forwarding at http://localhost:6006")
print("Logs: phoenix.log")


### What this cell does

- Launches `phoenix serve` as a background subprocess, redirecting stdout/stderr to `phoenix.log`.
- Keeps the notebook kernel free so you can continue running cells while Phoenix collects traces in the background.

### Opening the Phoenix UI

Once the server is up, head to **[http://localhost:6006](http://localhost:6006)** in your browser to access the Phoenix dashboard. Port-forwarding from the container to your local machine is handled automatically by the bootcamp environment — no additional setup needed.

You'll come back to this UI later in the notebook to inspect the traces generated by your agent runs.

> **Tip:** If the server fails to start, check `phoenix.log` in the working directory for errors. The most common cause is the port already being in use from a previous session.

### Reading Traces

Phoenix is already running on port **6006** with port-forwarding configured. Once the workflow runs below, a new trace will appear under the `movie-mcp` project at [http://localhost:6006/](http://localhost:6006/). Click into any trace to see:

- **Span tree (left)** — the nested execution: workflow → classify → movie_agent → LLM → MCP tool
- **Span details (right)** — chat messages rendered inline, tool arguments, JSON returns, token counts, latency, and stack traces on error
- **Project view** — aggregate latency / token / cost histograms and filterable search across every run

Common debugging patterns this unlocks:

- **Wrong answer?** Walk down the span tree until you find the first step whose output already contains the mistake — that span is the bug.
- **Slow response?** Sort spans by duration; LLM calls with oversized prompts or silent retry loops surface immediately.
- **Tool errors?** Filter spans by `status = error` to see every failed MCP call in the session.
- **Token blowout?** Project-level token aggregates make prompt regressions visible across runs without instrumenting anything yourself.

Everything we've built so far — the MCP tool server, the NIM-backed ReAct agent, the LangGraph intent classifier, and the Phoenix exporter — gets wired together in a single workflow YAML:

- **`function_groups.mcp_movies`** — the MCP server (tools)
- **`general.telemetry`** — Phoenix tracing, as configured above
- **`llms.nim_llm`** — the NVIDIA NIM backend
- **`functions.movie_agent`** — the ReAct agent, now declared as a **named function** so the LangGraph node can resolve it by name via `SyncBuilder.current().get_function("movie_agent")`
- **`workflow`** — the top-level `langgraph_wrapper` pointing at `intent_graph` in `movie_intent_classifier.py`

Notice what *isn't* in the YAML: any explicit instrumentation code. The same `movie_agent` that ran tracer-less in the earlier workflow now emits full Phoenix spans for every LLM and tool call just because `general.telemetry` is configured at the top of the file.


### Configuring Phoenix in the Workflow

Telemetry is just another section in the workflow YAML. The `general.telemetry` block activates the exporter for the entire run — every function, LLM, and tool inherits it automatically:

```yaml
general:
  telemetry:
    tracing:
      phoenix:
        _type: phoenix
        endpoint: http://localhost:${PHOENIX_PORT}/v1/traces
        project: movie-mcp
```

Three things worth noting:

1. **`_type: phoenix`** — the tracing provider plugin. NAT's telemetry layer is pluggable, so routing the same spans to a different OTel-compatible backend is a YAML-only change.
2. **`endpoint`** — Phoenix's OTLP-HTTP trace-ingest endpoint. The port is injected from `$PHOENIX_PORT` so the same YAML runs in this container, a laptop, and CI without edits.
3. **`project`** — a logical namespace. Every trace tagged `movie-mcp` lands in the same Phoenix project view, so you can diff runs, compare models, or isolate a regression by filtering on project.

In [ ]:
%%writefile movie_intent_workflow_tracing.yml                                                                                                                                                   
function_groups:                                                                                                                                                                        
    mcp_movies:                                                                                                                                                                           
      _type: mcp_client
      server:                                                                                                                                                                             
        transport: streamable-http
        url: "http://127.0.0.1:${MCP_PORT}/mcp/"

general:
  telemetry:
    tracing:
      phoenix:
        _type: phoenix
        endpoint: http://localhost:${PHOENIX_PORT}/v1/traces
        project: movie-mcp

llms:
    nim_llm:
      _type: nim
      model_name: nvidia/nemotron-3-nano-30b-a3b
      base_url: https://integrate.api.nvidia.com/v1
      api_key: ${NVIDIA_API_KEY}
      temperature: 0.0
      max_tokens: 1024

functions:
    movie_agent:
      _type: react_agent
      tool_names:
        - mcp_movies
      llm_name: nim_llm
      verbose: true
      parse_agent_response_max_retries: 3

workflow:
    _type: langgraph_wrapper
    graph: movie_intent_classifier.py:intent_graph

## Test the Workflow with Telemetry

Now both halves pay off together. We'll replay the **same three queries** from the "Probing the Agent's Boundaries" section earlier — two out-of-domain, one in-domain — but this time through the gated, traced pipeline. Open [http://localhost:6006/](http://localhost:6006/) (the `movie-mcp` project) in another tab and watch new traces appear as each cell runs.

Once you've seen the three baseline queries flow through, try your own queries to probe the agent's boundaries — edge cases you're curious about, ambiguous phrasings, or questions that sit right on the line between in and out-of-domain. The traces in Phoenix will show you exactly where the classifier drew the line and why.

In [ ]:
!nat run --config_file movie_intent_workflow_tracing.yml --input "How do I cook pasta?"

In [ ]:
!nat run --config_file movie_intent_workflow_tracing.yml --input "What is the capital of France?"

In [ ]:
!nat run --config_file movie_intent_workflow_tracing.yml --input "what is the rating of the movie The Dark Knight Rises?"

### Reading the Traces

Open the `movie-mcp` project in Phoenix and look at the traces for your own queries alongside the three baseline ones. The trace shape itself tells you how the classifier routed each query:

- **Rejected queries** (out-of-domain) have shape `workflow → classify → reject`. One short LLM call, no MCP span, sub-second latency. Check which of your queries ended up here — did the router agree with your expectations?
- **Accepted queries** (in-domain) have shape `workflow → classify → movie_agent → LLM(nim) → tool(search_movies) → MCP`. Multiple LLM calls (one for classification, one or more for the ReAct loop), at least one MCP span, and a noticeably larger token budget.

The ambiguous queries are the interesting ones — open those traces and read the classifier's output. That single word is the whole decision. If it routed somewhere you didn't expect, the fix is almost always in the classifier's system prompt, not the agent.

## Optional Challenge: Cover All Tables

The `search_movies` tool you built only touches the **IMDB** table — and even there, it only surfaces 6 of IMDB's 51 columns. There's a lot of information the agent can't see:

- **`earning`** — `Domestic` and `Worldwide` box office per movie
- **`genre`** — many-to-many mapping across 20 genres (Drama, Sci-Fi, Biography, Action, …)
- **The 45 IMDB columns we skipped** — `MetaCritic` score, and demographic vote breakdowns: by age bracket (`VotesU18`, `Votes1829`, `Votes3044`, `Votes45A`), by gender (`VotesM`, `VotesF`), by region (`VotesUS`, `VotesnUS`), and raw vote counts per demographic (`CVotes*`).

The agent currently can't answer questions like *"What's the worldwide gross of Inception?"*, *"Which biography has the highest rating?"*, or *"Which movie is most loved by viewers under 18?"* — each needs a table or column the tool doesn't expose. This section is **optional** — try it if you'd like to extend the system on your own.

### Sample Question to Target

> **"Which drama has the highest worldwide box office?"**

To answer this, the agent needs to find all `Drama` movies (`genre` table), look up their worldwide gross (`earning` table), and resolve the winner back to its title (`IMDB` table) — a three-way cross that the current tool can't do.

### More Questions to Try

Pick a few that stretch different parts of the schema:

| Question | What it exercises |
|----------|-------------------|
| *"What's the worldwide gross of Inception?"* | IMDB (title → id) + `earning` |
| *"Which biography has the highest IMDB rating?"* | IMDB + `genre` |
| *"What's the longest movie rated 8.5 or higher?"* | IMDB `Runtime` + `Rating` (note: `Runtime` is stored as text like `"148 min"`) |
| *"Which movie has the biggest rating gap between male and female viewers?"* | IMDB `VotesM`, `VotesF` |
| *"Which movie is most loved by viewers under 18?"* | IMDB `VotesU18` |
| *"Which movie has the biggest disagreement between critics and audiences?"* | IMDB `MetaCritic` vs `Rating` (remember MetaCritic is 0–100, Rating is 0–10) |
| *"Which comedy had the best return on investment (worldwide gross ÷ budget)?"* | all three tables + arithmetic |

Each row forces the agent to call at least one capability it doesn't have today — you decide which ones to build.

### Steps to Implement

1. **Extend `MovieDB`** in `movie_db.py` with methods that query `earning` and `genre`.
2. **Register the new tools** in `movie_server.py` — add `types.Tool` entries in `handle_list_tools` (clear `description` and `inputSchema` matter, that's what the LLM reads), plus matching dispatch branches in `handle_call_tool`.
3. **Restart the MCP server** — the running process caches the old tool list, so `server_process.terminate()` and then re-run the `subprocess.Popen` cell.
4. **Verify** discoverability with `!nat mcp client tool list --url http://127.0.0.1:8000/mcp/` — your new tools should show up with their descriptions.
5. **Re-run** the workflow with one of the questions above and inspect the Phoenix trace — you should see the new tool calls chained in the span tree, proving that the agent is doing the cross-table reasoning you wired up, not hallucinating.

### Cleanup

Stop all the processes

In [ ]:
# Kill MCP Server
mcp_server_process.terminate()
mcp_server_process.wait()

# Kill Phoenix
phoenix_process.terminate()
phoenix_process.wait()

## Use Cases

The NeMo Agent Toolkit isn't an abstract framework — it's the foundation under real systems solving real problems. To make that concrete, we'll walk through two very different use cases that both sit on top of NAT, showing how the same toolkit scales from a single focused agent to a full multi-agent enterprise research platform.

### AI-Q

The **[AI-Q Blueprint](https://github.com/NVIDIA-AI-Blueprints/aiq)** is an open, enterprise-grade reference for building intelligent research agents that connect to your organization's data, reason over it with state-of-the-art models, and return **citation-backed answers** — not just plausible-sounding prose. 

Built on top of the NVIDIA NeMo Agent Toolkit and LangGraph, the latest [**v2.0**](https://github.com/NVIDIA-AI-Blueprints/aiq/releases/tag/2.0.0) release (March 2026) is a ground-up rewrite that introduces a **two-tier multi-agent architecture**: every incoming query first hits a lightweight **Intent Classifier** that decides whether it's a meta question (answered instantly), a **Shallow Research** task (bounded, tool-calling agent optimized for speed), or a **Deep Research** task (a multi-phase workflow with a dedicated Orchestrator, Planner, and Researcher, plus an optional human-in-the-loop Clarifier that generates and approves a research plan before kicking off a long run).

<div style="text-align: center;">
  <img src="images/aiq_arch.png" style="width: 600px; height: auto;">
</div>


What makes [AI-Q](https://docs.nvidia.com/aiq-blueprint/latest/index.html) production-ready is that it's a **direct, end-to-end implementation of the NVIDIA NeMo Agent Toolkit (NAT)** — every piece of the system is defined, wired, and operated through NAT's primitives. The agents, LLMs, tools, and routing logic all live in **NAT YAML configuration files** with environment variable substitution, so you can reassign roles (e.g., a frontier GPT-5.2 as the orchestrator with open-source Nemotron researchers underneath) without touching a line of Python. 

<div style="text-align: center;">
  <img src="images/aiq.png" style="width: 600px; height: auto;">
</div>

The whole lifecycle runs through NAT's CLI — `nat run` for interactive execution, `nat serve` for the API, and `nat eval` for the built-in evaluation harness that ships with FreshQA and DeepResearch Bench. 

Around this NAT-driven core, AI-Q layers the enterprise plumbing you'd otherwise have to build yourself: an **async Jobs API** with SSE streaming and event replay, **Dask-based distributed execution** for long-running deep research jobs, a **pluggable knowledge layer** (swap LlamaIndex for NVIDIA's Foundational RAG without changing agent code), multimodal PDF extraction via VLMs, **PostgreSQL persistence** for job state and LangGraph checkpoints, and a **deterministic citation verification pipeline** that validates every claim against actually-retrieved sources. 

The result is a [blueprint](https://github.com/NVIDIA-AI-Blueprints/aiq) that currently holds **top positions on both the [DeepResearch Bench](https://huggingface.co/spaces/muset-ai/DeepResearch-Bench-Leaderboard) and [DeepResearch Bench II](https://agentresearchlab.com/benchmarks/deepresearch-bench-ii/index.html#leaderboard) leaderboards** while shipping with Docker Compose stacks, Helm charts, a Next.js frontend, and a three-part Jupyter notebook tutorial. If DABStep shows what NAT looks like when a single agent builds its own reusable tools, AI-Q shows what NAT looks like at the other end of the spectrum — a multi-agent research system with auth, persistence, observability, and evaluation, ready to put in front of real users. 


### NVIDIA KGMON Data Explorer

The **NVIDIA KGMON Data Explorer** is a real-world use case built on top of the NeMo Agent Toolkit that tackles one of the hardest problems in agentic AI: multi-step reasoning over structured tabular data. Most "Deep Research" agents rely on web search, but in domains like finance, healthcare, or operations, the answers live in CSVs, JSONs, and dense domain manuals — not on the internet. 

<div style="text-align: center;">
  <img src="images/data_explorer.png" style="width: 600px; height: auto;">
</div>

The team benchmarked their approach on [DABStep](https://huggingface.co/spaces/adyen/DABstep) (Data Agent Benchmark for Multi-step Reasoning), a 450-task benchmark from Adyen focused on the financial payments sector, where 84% of tasks require complex cross-referencing between heterogeneous data sources and domain rules. Their key insight was architectural rather than model-driven: instead of having one heavyweight model solve every question from scratch, they split the system into three phases — a **Learning Loop** (heavy model builds a reusable `helper.py` library from training tasks), a **Fast Inference** loop (lightweight model orchestrates those pre-built tools), and an **Offline Reflection** phase (heavy model reviews outputs and injects insights back into the inference prompt). 

<div style="text-align: center;">
  <img src="images/data_explorer_arch.png" style="width: 600px; height: auto;">
</div>

The result: **1st place on DABStep**, a 30x speedup over the Claude Code + Opus 4.5 baseline, and — most strikingly — a smaller model (Haiku 4.5) beating a larger one (Opus 4.5) by **23 points on hard tasks** (89.95 vs 66.93). The takeaway for anyone building agents: the ceiling is often architectural, not model-sized. Read the Full blog on [HuggingFace](https://huggingface.co/blog/nvidia/nemo-agent-toolkit-data-explorer-dabstep-1st-place)


## Links and Resources

- [NeMo Agent Toolkit Documentation](https://docs.nvidia.com/nemo/agent-toolkit/latest/index.html)
- [NeMo Agent Toolkit GitHub](https://github.com/NVIDIA/NeMo-Agent-Toolkit)
- [Arize Phoenix](https://phoenix.arize.com/)
- [Arize Phoenix Github](https://github.com/arize-ai/phoenix)

---

## Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.

<p> <center> <a href="../start_here.ipynb">Home Page</a> </center> </p>

<div>
    <span style="float: left; width: 33%; text-align: left;"><a href="04_langraph_agent.ipynb">Previous Notebook</a></span>
    <span style="float: left; width: 34%; text-align: center;">
        <a href="01_inference_endpoint.ipynb">1</a>
        <a href="02_introduction_mcp.ipynb">2</a>
        <a href="03_low_level_mcp.ipynb">3</a>
        <a href="04_langraph_agent.ipynb">4</a>
        <a >5</a>
        <a href="06_challenge.ipynb">6</a>
        <a href="bonus_challenge/07_bonus_challenge.ipynb">7</a>
    </span>
    <span style="float: left; width: 33%; text-align: right;"><a href="06_challenge.ipynb">Next Notebook</a></span>
</div>